# 课程总览：什么是 AI Agent


> 我们已经知道，大语言模型能生成流畅的文本，能写代码、做翻译、回答知识性问题。但单次调用只输出一段文本，不接触外部世界，也无法把一个需要多步操作的任务从头做到尾。
>
> 这一讲是整门课的起点，回答三个问题：AI Agent 是什么，为什么它是大语言模型的下一个形态，这门课将按什么路线把 Agent 亲手做出来。我们会先看一个单次调用做不到、Agent 循环能做到的任务，再从零搭出第一个最小循环。


Agent 没有一个统一的定义，但这一讲可以确立一个贯穿全课的说法：Agent 是一个循环，LLM 根据当前状态决定下一步动作，动作在环境中执行，观察结果再喂回 LLM，直到任务完成。这句话里的每个环节都对应一个具体组件，后面五讲会逐一展开。

先把这个说法里的三个词分别说清楚。环境是 Agent 身外的世界：文件系统、网页、数据库、代码执行器都算。状态是任务进行到当前的全部信息，包括已经做过的动作和已经拿到的中间结果。动作是模型输出的、能在环境里真实执行的一步操作，比如编辑一个文件或运行一条命令。放到"把仓库里所有过期的 TODO 标记出来，并运行测试"这个任务上：环境是仓库和测试命令，状态是已扫描的文件与已标记的过期条目，动作是编辑文件、运行测试。

把它交给一个只有对话能力的大语言模型，它会给出建议文本，但不会真的去改文件。要让模型完成这类任务，需要把单次问答改造成循环执行——模型负责思考和选动作，我们负责维护状态、调用工具、把环境结果带回去。这一讲从单次调用与 Agent 的差异出发，给出定义，从零实现一条最小循环，再看主流范式与课程路线。

## 1. 从 LLM 应用到 Agent

单个 forward pass 的 LLM 有四条硬边界。

一是只能"说"不能"做"：它输出 token，不接触世界，不知道网页、数据库、文件或代码执行器里发生了什么。二是上下文窗口有限：长任务和大量中间结果无法在一次调用内全部容纳。三是错了不会自己纠正：单次生成没有执行反馈，无法知道答案是否正确。四是知识静态：训练截止日期之后的新事实无法获取。

先建立统一的客户端，再发起一次单次调用，观察它的输出。


In [ ]:
# 把仓库根目录加入模块搜索路径，才能 import 到 llm_client
import sys
import os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)

from llm_client import get_llm

# force_mock=True 保证离线可复现；配置真实 key 后去掉该参数即可
client = get_llm()
{"client": type(client).__name__, "mock": client.is_mock}


In [ ]:
# 一个 LLM 应用的核心动作：把任务拼进 prompt，调用一次，拿到文本
task = "把仓库里所有过期的 TODO 标记出来，并运行测试"
reply = client.chat([{"role": "user", "content": task}])
# 只保留足以观察行为的摘要，避免把整段模型回复当作实验结果。
{"reply_preview": reply[:160], "reply_chars": len(reply),
 "tool_calls": 0}


上面列出的四条硬边界，我们逐条展开，用具体的数字和例子来看。

**只能"说"不能"做"**。模型的全部输出是 token，也就是文本。它不能真的打开文件、发送请求或执行代码；这些操作只能由我们写的程序替它完成。让模型"把过期的 TODO 标记出来"，它只会输出一段描述怎么改的文字，改动本身要由外部的脚本执行。

**上下文窗口有限**。上下文窗口（context window）是模型一次调用里能读取的最大 token 数，包括输入的 prompt 和输出的内容，两者加起来不能超过这个数。token 是文本的切分单位，中文大约一个字对应一个 token，英文一个常见单词约对应一个 token。GPT-2 的窗口是 1024 个 token，约一千字；现代主流模型在 128K 到 1M 之间。判断一次调用能否容纳一个任务，先把任务换算成 token：一行代码平均约 20 个 token，一个 3000 行的仓库约 6 万 token，还没算模型输出，就已经接近 128K 窗口的一半。任务越长、中间结果越多，越容易超出窗口，这就是长任务必须拆成多步、分批喂给模型的原因。

**错了不会自己纠正**。单次调用里模型只生成一次答案，中间没有任何执行反馈。它写了一段有 bug 的代码，却不知道代码能否跑通，因为没有人运行它。纠正需要反馈：运行代码、把报错信息喂回去、让模型在下一轮重写。

**知识静态**。模型在训练时见过一份固定的数据，数据截止日期之后的事实它不知道。某个依赖库在 2025 年改了接口，而训练数据截止于 2024 年，模型给出的调用方式就过时了。要拿到新事实，只能靠外部检索再喂回模型，这正好是 Agent 循环里工具要承担的工作。

前两条是结构限制：输出不能执行、窗口容纳不了。后两条是信息限制：没有反馈、知识旧。Agent 循环的每一环都在补其中一条：动作交给工具执行，反馈由循环带回来，新知识靠外部检索获取。

单次调用与 Agent 的差别，可以用一张表来对照。

| 维度 | 纯 LLM 应用 | LLM-based Agent |
|:---|:---|:---|
| 调用方式 | 单次（或固定轮次）问答 | 循环，直到终止条件 |
| 状态 | 无（或仅靠会话历史） | 显式维护任务状态与记忆 |
| 动作 | 只输出 token | 输出可执行的工具调用 |
| 环境反馈 | 无 | 有（执行结果回到上下文） |
| 目标 | 生成一个答案 | 完成一个任务（可失败、可重试） |

表里的状态、动作、工具调用，落在 TODO 任务上都很具体。状态是任务进行到当前的记录：扫过了哪些文件、哪几处 TODO 已标记为过期。动作是模型输出的、能被执行的一步操作，比如"把文件 src/main.py 里第 12 行的 TODO 标为过期"。工具调用是动作的一种具体形式：模型输出一段结构化文本，比如 search("CS329A agents")，程序解析这段文本并调用对应的函数。动作能不能执行，取决于循环里有没有注册对应的工具。

有一个容易混淆的地方：在代码里自动拼一个 prompt 再调一次 API，只是把 LLM 当函数调用，没有循环、状态与动作，那就仍是应用而不是 Agent。判断的关键是，循环、状态、动作这三样缺一不可。

## 2. Agent 的定义与组成

Agent 的经典定义先于大语言模型存在。Wooldridge 在 1995 年的综述里给出：agent 是居于某个环境中的实体，通过传感器感知环境，通过执行器作用于环境。定义里有两点在后续讨论里反复出现。一是 autonomy（自主性）：在没有外部直接干预的情况下，agent 自行决定行动以达成目标。二是 situatedness（具身性）：它面对的是一个部分可观、不断变化的环境，而不是一个格式良好的输入。

这些词放在软件里都有具体对应。传感器是 Agent 获取环境信息的入口：对人来说是眼睛和耳朵，对软件 Agent 来说是读文件、检索网页、查询数据库。执行器是 Agent 改变环境的手段：对人来说是手和脚，对软件 Agent 来说是修改文件、发送请求、运行命令。自主性强调动作由 Agent 自己选：给定"整理仓库 README"的目标，它自己决定先读哪些文件、按什么顺序改，而不是每一步都由人指定。部分可观指 Agent 任何时候都只能看到环境的一部分：扫过文件列表的程序，不知道某个文件内部有哪些过期的 TODO。而"格式良好的输入"说的是普通程序：输入结构固定、字段齐全，程序从头算到尾，中途不需要再看外部世界。Agent 面对的是信息不全、随时变化的环境，所以需要反复感知和行动。

LLM-based Agent 是现代版本：以 LLM 为决策核心，在循环中调用 LLM 完成感知、推理、决策，把决策变成环境中的动作，通常借助工具，再把动作结果作为反馈再次输入，直到目标达成。LLM 在这里扮演大脑，而不是全部。循环的最小组成有五项，先用一段代码把它们列出来。

In [ ]:
# Agent 循环的最小组成，用列表摆在一起
components = [
    ("感知", "把环境状态与工具返回组装成上下文"),
    ("决策", "LLM 基于状态与目标输出下一步动作"),
    ("行动", "解析动作，在工具上执行"),
    ("反馈", "执行结果回到上下文，作为下一轮感知"),
    ("终止", "模型给出 Final Answer，或步数用尽"),
]
for name, desc in components:
    print(f"{name} —— {desc}")


循环之外还有几个常被讲成独立模块、其实是循环配件的部分。

记忆负责把上下文窗口之外的信息接回来：短期是消息历史，也就是本次对话里所有消息的列表；长期是外部存储（向量库或文件），比如把上一会话的结论写进一个文件，下次会话再读回来。规划把目标分解成子目标，或对动作序列做搜索：任务"写一份课程笔记"，先拆成"查大纲、检索资料、写正文、校对"几步，再逐步执行；搜索则是一次尝试多条动作路线，保留能达成目标的那条。验证在循环之外加一个检查环节，让输出经过验证才被接受：模型生成的代码先跑一遍测试，通过了才作为最终答案。它们不是循环的主体，但都挂接在循环的某个环节上，后面讲到对应讲次时再展开。

说它们是配件而不是主体，是因为缺了它们循环照样能转：一个只有 LLM、工具和消息历史的循环，可以完整跑完一个小任务。前面要实现的这条最小循环就属于这种情况。

循环的主体只有四步加一个终止条件，我们从零实现它。

## 3. 一条 Agent 循环

把上一节的组成写成可运行的代码，是这一讲的第一个动手目标。循环体做四件事：调用 LLM 得到回复，解析回复里的动作，执行动作并把观察结果回填进消息历史，重复直到模型给出 `Final Answer` 或步数用尽。消息历史就是循环的状态。

消息历史是循环里维护的一张消息列表，每条消息记着一个角色和一段内容，形如 {"role": "user", "content": "..."}。开始时的第一条是任务本身；之后每轮，Agent 的输出以 assistant 身份追加，工具的执行结果以 user 身份追加。模型每轮只看这张列表：自己上一轮说了什么、工具返回了什么，全部要靠列表里的记录。列表越长，模型能参考的上下文就越多，所以消息历史就是循环的状态。

任务选一个多步算术：计算 (3+5)×(7-2)。先在纸上分步算一遍，给代码一个对照基准。

In [ ]:
# 手算验证：不借助工具，先把 (3+5)×(7-2) 分步算一遍
step1 = 3 + 5
step2 = 7 - 2
step3 = step1 * step2
print(f"第一步 (3+5) = {step1}")
print(f"第二步 (7-2) = {step2}")
print(f"第三步 8×5   = {step3}")
print("手算与之后 Agent 循环的结果应当一致")


把这段历史在纸上逐步写出来，任务固定为"请分步计算 (3+5)×(7-2)"。

第一步，消息列表只有任务这一条：

```text
[用户] 请分步计算 (3+5)×(7-2)
```

第二步，模型输出 Action: calc("3+5")，以 assistant 身份追加；工具执行后，结果以 user 身份回填。第三步与第四步重复同样的过程：模型先输出 Action: calc("7-2") 拿到 5，再输出 Action: calc("8×5") 拿到 40。到第四步结束，整个列表是：

```text
[用户] 请分步计算 (3+5)×(7-2)
[Agent] Action: calc("3+5")
[用户] Observation: 8
[Agent] Action: calc("7-2")
[用户] Observation: 5
[Agent] Action: calc("8×5")
[用户] Observation: 40
```

每一步模型读到的列表都比上一步多两条消息。最后一步，模型看到列表里的 8、5、40，输出 Final Answer: 40。这个 40 是模型从列表里读出来的，不是它自己心算的——工具结果经过消息列表回到模型，这就是状态在循环里的流动。

模型输出的文本需要先被解析成结构化动作。模型可能输出三种内容：`Thought` 加 `Action`、`Final Answer`、或普通文本。解析器负责识别这三种情况，返回统一的 (kind, action, payload) 结构。

三种内容各举一个例子。第一种是推理加动作，模型一边想一边指定要调用的工具：

```text
Thought: 需要先检索课程资料。
Action: search("CS329A agents")
```

Action 后面是工具名和参数。解析器要把这行提取成结构化的 (工具名, 参数)，循环才知道该调用哪个函数。第二种是最终答案，表示任务完成：

```text
Final Answer: CS329A 是斯坦福的 AI Agent 课程
```

第三种是普通文本，既没有 Action 也没有 Final Answer，比如模型在复述任务。解析器对这类文本返回"无动作"，循环就不调用工具。

解析的顺序有讲究：先找 Final Answer，再找 Action。模型可能在同一次回复里既给出 Action 又给出 Final Answer，此时循环应该先把动作执行完，再按最终答案终止。解析器返回的 (kind, action, payload) 三个字段分别回答三个问题：是否终止、执行哪个工具、最终答案是什么。

In [ ]:
import re


def parse_response(text):
    """把模型的回复解析成结构化动作。

    返回 (kind, action, payload)：
    - kind: "final" / "action" / "idle"
    - action: (工具名, 参数) 或 None
    - payload: 最终答案（kind 为 final 时），否则为 None
    回复里同时出现 Action 与 Final Answer 时，kind 取 final，
    但 action 仍然返回，循环可以先执行动作、再按 final 终止。
    """
    final = re.search(r"Final Answer:\s*(.+)", text, re.DOTALL)
    action = re.search(r"Action:\s*(\w+)\s*\((.*?)\)", text, re.DOTALL)
    action_pair = (action.group(1), action.group(2).strip()) if action else None
    if final:
        return ("final", action_pair, final.group(1).strip())
    if action:
        return ("action", action_pair, None)
    return ("idle", None, None)


# 用几段文本测试解析器：只有 Action、只有 Final、普通文本、两者同时出现
samples = [
    'Thought: 先搜索。\nAction: search("CS329A agents")',
    "Thought: 检索完成。\nFinal Answer: 这是结论",
    "模拟回复：这只是一段普通文本",
    'Thought: 有动作也有结论。\nAction: search("x")\nFinal Answer: 结论',
]
for text in samples:
    kind, action, payload = parse_response(text)
    print(f"kind={kind!r}  action={action}  payload={payload!r}")


In [ ]:
class AgentLoop:
    """最小 Agent 循环：感知 → 决策 → 执行 → 反馈，直到终止。

    参数：
    - brain：可调用对象，接收消息历史，返回模型输出的文本
    - tools：字典，工具名到可调用对象的映射
    """

    def __init__(self, brain, tools):
        self.brain = brain
        self.tools = tools
        self.messages = []

    def execute_tool(self, name, args):
        """在工具注册表里查找并执行工具，返回结果字符串。"""
        if name not in self.tools:
            return f"Error: 未知工具 {name}"
        clean_args = args.strip().strip("\"'")
        return str(self.tools[name](clean_args))

    def step(self):
        """执行一轮：调用大脑、解析动作、执行工具。返回 (是否终止, 执行过的动作名)。"""
        reply = self.brain(self.messages)
        self.messages.append({"role": "assistant", "content": reply})
        kind, action, _ = parse_response(reply)
        executed = []
        if action is not None:
            name, args = action
            result = self.execute_tool(name, args)
            self.messages.append({"role": "user", "content": f"Observation: {result}"})
            executed.append(name)
        has_final = kind == "final"
        return has_final, executed

    def run(self, task, max_steps=10):
        """运行循环直到模型给出 Final Answer 或步数用尽，返回完整的消息历史。"""
        self.messages = [{"role": "user", "content": task}]
        for _ in range(max_steps):
            has_final, _ = self.step()
            if has_final:
                break
        else:
            self.messages.append({"role": "user",
                                  "content": "Error: 步数用尽，未给出 Final Answer"})
        return self.messages


In [ ]:
def calc(expr):
    """执行一个单步算术表达式，支持一次加、减、乘。"""
    expr = expr.strip()
    for op, fn in [("+", lambda a, b: a + b),
                   ("-", lambda a, b: a - b),
                   ("×", lambda a, b: a * b)]:
        if op in expr:
            left, right = expr.split(op, 1)
            return fn(int(left), int(right))
    raise ValueError(f"无法解析表达式: {expr}")


def arithmetic_brain(messages):
    """脚本化算术大脑：按固定计划调用 calc，从观察里读取中间结果。

    真实运行时，这一步由 llm_client 的模型完成；这里用固定轨迹保证离线可跑。
    """
    issued = " ".join(m["content"] for m in messages)
    plan = ['calc("3+5")', 'calc("7-2")', 'calc("8×5")']
    for action in plan:
        if action not in issued:
            return f"Thought: 中间状态已回填，继续执行计划。\nAction: {action}"
    last_obs = [m for m in messages if m["content"].startswith("Observation")]
    final = last_obs[-1]["content"].split(": ", 1)[1] if last_obs else "40"
    return f"Final Answer: {final}"


tools = {"calc": calc}
task = "请分步计算 (3+5)×(7-2)，每一步用一次 calc 调用，最后给出 Final Answer。"
loop = AgentLoop(brain=arithmetic_brain, tools=tools)
trace = loop.run(task, max_steps=6)
for msg in trace:
    tag = "用户" if msg["role"] == "user" else "Agent"
    print(f"[{tag}] {msg['content']}")
    print()
print("关键观察：中间结果 8、5、40 由我们回填进消息历史，循环靠它逐步推进")


有一个常见误解需要澄清：Agent 循环不等于多轮对话。

ChatGPT 的多轮只是记住轮次，模型每次仍在生成文本，没有动作，也没有反馈。Agent 循环的每一轮一定经过环境，哪怕只是读一个文件。判断标准是中间有没有可执行的、改变世界的动作。上一节里，`calc` 就是这样的动作：它真的执行了算术并返回结果。

把两种轨迹并排看。多轮对话的轨迹是用户一句、模型一句，列表里没有 Observation；Agent 循环的轨迹是"模型输出 Action，程序执行工具，把 Observation 喂回去"，列表里至少出现一条来自工具的记录。读一个文件也算动作，因为它确实从环境读回了内容，这段内容进入了模型下一轮的上下文。反过来，模型输出一段不含 Action 的文字，就只是多轮对话里的一轮。

我们把同一个循环的大脑换成 llm_client 的 mock 客户端，看循环如何处理模型输出的脚本化轨迹。

In [ ]:
# 同一段循环代码，大脑换成 llm_client 的 mock，行为随之变化
search = lambda q: f"检索到与 {q} 相关的资料 3 条"
tools = {"search": search}

task = ("请检索 CS329A 的课程信息并给出结论。\n"
        "每一步请使用如下格式：\n"
        "Thought: ...\n"
        "Action: search(\"关键词\")\n"
        "或给出 Final Answer: 结论")

loop = AgentLoop(brain=client.chat, tools=tools)
trace = loop.run(task, max_steps=3)
for msg in trace:
    tag = "用户" if msg["role"] == "user" else "Agent"
    print(f"[{tag}] {msg['content']}")
    print()
print("关键观察：mock 把 Action 与 Final Answer 放在同一次回复里，循环先执行动作再终止")


In [ ]:
# 用两个数字量化"单次调用 vs 循环"的差距
def count_tool_calls(messages):
    """统计消息历史里执行过的工具动作数（以 Observation 消息计）。"""
    return sum(1 for m in messages if m["content"].startswith("Observation"))


single_context = len(reply)
loop_context = sum(len(m["content"]) for m in trace)
print(f"单次调用：上下文 {single_context} 字符，工具调用 0 次")
print(f"Agent 循环：上下文 {loop_context} 字符，工具调用 {count_tool_calls(trace)} 次")
print("关键观察：能力增量来自循环与工具反馈，而不是模型本身")


## 4. Agent 生态地图

围绕 Agent 循环，业界形成了几种主流范式。它们不是互斥的分类，而是"决策怎么来"的几种方案，真实系统通常是多种范式的组合。我们把六种范式、各自的代表工作，以及它们在本课中的位置列出来。


In [ ]:
# 六种主流范式的速查卡片，以及两个真实场景的选型练习
paradigms = {
    "ReAct": ("推理与行动交替，把想和做织进一条轨迹", "Yao et al. 2022", "L4"),
    "Tool use": ("模型输出结构化工具调用，工具结果作为输入", "Toolformer / MCP", "L4"),
    "Planning": ("先分解任务，或对动作空间做搜索", "LATS", "L5"),
    "Multi-agent": ("多个 Agent 协作、辩论、分工", "AutoGen", "L6/L7"),
    "Memory-based": ("显式长期记忆，跨会话与长任务工作", "MemGPT", "L11"),
    "Frameworks": ("把循环、工具、记忆封装成库", "LangGraph / Claude SDK", "全程参照"),
}
for name, (desc, work, pos) in paradigms.items():
    print(f"{name:<14} {desc}  |  {work}  |  {pos}")

print()
scenarios = [
    ("读多个文件、跑测试并逐轮修 bug 的编程助手", "ReAct + Tool use"),
    ("先列大纲、再逐节检索资料并写作的长报告", "Planning + Memory"),
    ("一位老师和一个学生轮流出题答题，用来生成训练数据", "Multi-agent"),
]
for scene, answer in scenarios:
    print(f"场景：{scene}")
    print(f"  合适范式：{answer}")
print("关键观察：范式不互斥，一个真实 Agent 常同时是 ReAct + Planning + Memory")


六种范式里，ReAct 与 Planning 最贴近循环本身，先看懂这两个，其余四种用一句话说明。

ReAct 是推理与行动交替：模型每轮先输出一段 Thought 说明为什么这样做，再输出一个 Action 指定要调用的工具，环境返回 Observation 后进入下一轮。前一条 Agent 循环就是这个形状——Thought、Action、Observation 交替出现，只是我们的解析器没有把 Thought 单独取出来。Planning 是动手前先规划：把目标分解成子目标，或对动作序列做搜索。分解的例子是"写一份课程笔记"，先列成"查大纲、检索资料、写正文、校对"四步再逐步执行；搜索是同时尝试多条动作路线，保留能达成目标的那条，LATS 属于这类工作。ReAct 边看边做，Planning 先计划后行动。

其余四种各自举一个例子。Tool use 让模型输出结构化工具调用，Toolformer 是早期代表，今天的 MCP 是这类调用的标准化接口。Multi-agent 让多个 Agent 对话协作，一位老师和一个学生轮流出题答题，就是典型用法。Memory-based 用显式长期记忆跨会话工作，MemGPT 用分层记忆管理长对话：超出上下文的内容先转存到外部存储，需要时再取回。Frameworks 把循环、工具、记忆封装成库，LangGraph 和 Claude SDK 属于这一类，全程可以参照。

真实系统很少只用一种范式：以 ReAct 做主干，用 Planning 分解长任务，用 Memory 记住历史，用 Tool use 接外部工具，是常见的组合方式。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 6.5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6.5)
ax.axis("off")
ax.set_title("The Agent Loop", fontsize=13)

# 主循环四个环节（图内文字用英文）
nodes = [
    (5.0, 5.6, "Observation"),
    (8.5, 3.0, "Reason + Action"),
    (5.0, 0.4, "Execution"),
    (1.5, 3.0, "Feedback"),
]
for x, y, label in nodes:
    box = mpatches.FancyBboxPatch((x - 1.5, y - 0.6), 3.0, 1.2,
                                  boxstyle="round,pad=0.08",
                                  fc="#e8eefb", ec="#4a6fa5", lw=1.4)
    ax.add_patch(box)
    ax.text(x, y, label, ha="center", va="center", fontsize=11)

# 顺时针循环箭头
def arrow(p1, p2):
    ax.annotate("", xy=p2, xytext=p1,
                arrowprops=dict(arrowstyle="-|>", color="#333333", lw=1.8))

arrow((5.0, 4.9), (7.0, 3.7))   # Observation -> Reason + Action
arrow((7.1, 2.6), (6.2, 1.0))   # Reason + Action -> Execution
arrow((3.8, 1.0), (2.9, 2.6))   # Execution -> Feedback
arrow((3.0, 3.7), (3.5, 4.9))   # Feedback -> Observation

# 配件：挂到对应环节，标注讲次
accessories = [
    (0.5, 5.6, "Verification (L3)"),
    (9.5, 5.6, "Memory (L11)"),
    (9.5, 0.4, "Planning (L5)"),
    (0.5, 0.4, "Training (L6-L9)"),
]
for x, y, label in accessories:
    box = mpatches.FancyBboxPatch((x - 1.1, y - 0.4), 2.2, 0.8,
                                  boxstyle="round,pad=0.05",
                                  fc="#f7f2e6", ec="#a58a4a", lw=1.2)
    ax.add_patch(box)
    ax.text(x, y, label, ha="center", va="center", fontsize=9)

plt.tight_layout()
plt.show()


## 5. 课程路线

课程沿四部分展开，共 17 个 notebook。Part 1 补能力：test-time compute、验证、工具、规划，从"LLM 只能生成"出发逐层加。Part 2 补怎么变强：训练期缩放、开放进化、搜索、后训练演进。Part 3 补工程化：SWE、记忆、评测。Part 4 补边界：推理、数学、自治、机器人。我们把每部分的讲次列出来。


In [ ]:
# 课程地图：四部分、17 个 notebook
roadmap = [
    ("Part 1  Foundation",
     ["L1 课程总览", "L2 Test-time compute", "L3 鲁棒验证",
      "L4 工具与代码反馈", "L5 多步规划"]),
    ("Part 2  Training",
     ["L6 训练期缩放", "L7 开放进化", "L8 搜索与深度研究", "L9 后训练演进"]),
    ("Part 3  Engineering",
     ["L10 SWE Agent", "L11 记忆系统", "L14 评测"]),
    ("Part 4  Frontiers",
     ["L12 LLM 推理", "L13 数学 Agent", "L15 自治 Agent",
      "L16 多模态机器人", "L17 未来方向"]),
]
for part, lectures in roadmap:
    print(part, "—", "、".join(lectures))
total = sum(len(lectures) for _, lectures in roadmap)
print(f"\n总计 {total} 个 notebook")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

parts = ["Part 1: Foundation", "Part 2: Training",
         "Part 3: Engineering", "Part 4: Frontiers"]
counts = [5, 4, 3, 5]
colors = ["#4a6fa5", "#6a9fb5", "#9ab0a5", "#b58a6a"]

fig, ax = plt.subplots(figsize=(8, 4))
y = np.arange(len(parts))
ax.barh(y, counts, color=colors, edgecolor="#333333")
for i, c in enumerate(counts):
    ax.text(c + 0.08, i, str(c), va="center", fontsize=11)
ax.set_yticks(y)
ax.set_yticklabels(parts, fontsize=10)
ax.set_xlabel("Number of notebooks", fontsize=10)
ax.set_title("Course Roadmap: 4 Parts, 17 Notebooks", fontsize=12)
ax.set_xlim(0, 6.5)
plt.tight_layout()
plt.show()


每一讲都沿同一条教学契约推进：直觉理解 → 手算验证 → 代码实现 → 实验观察。这一讲已经示范了前面几步：先建立循环的直觉，再在纸上分步推演，最后从零把它写出来。后续每一讲都会重复这个节奏。

到这一讲结束，我们已经写出了第一个最小循环：一个 `AgentLoop` 类、一个工具注册表、一个能解析动作的解析器，加起来不到五十行。这足以说明一件事：Agent 与应用的差距在结构而不在模型。循环把环境、工具、记忆和多步算力接入模型，能力增量由此而来。

下一讲从 test-time compute 开始，讨论训练完成后把额外的算力投入推理阶段，能在多大程度上补偿单次生成的质量。


## 小结

这一节所学的内容：

- [ ] Agent 是循环：LLM 根据当前状态决定下一步动作，动作在环境中执行，观察结果再喂回 LLM，直到任务完成
- [ ] 单次调用的 LLM 有四条硬边界：不能做、上下文有限、不会自纠、知识静态
- [ ] 上下文窗口是模型单次调用能读取的最大 token 数，超过的部分无法进入模型视野
- [ ] 在代码里调 LLM 不是 Agent，循环、状态、动作三样缺一不可
- [ ] 循环的状态由消息历史承载：每轮回复与工具观察都被记录，模型只能看到列表里的内容
- [ ] Agent 循环的最小组成：感知、决策、行动、反馈、终止
- [ ] 自主性指无人干预也能行动，具身性指面对部分可观、不断变化的环境
- [ ] 记忆、规划、验证是循环的配件，不是主体
- [ ] Agent 循环不等于多轮对话，判断标准是中间有没有改变世界的动作
- [ ] 六种范式不互斥，真实 Agent 常是多种范式的组合
- [ ] 课程沿四部分展开，每个概念按直觉理解 → 手算验证 → 代码实现 → 实验观察 推进

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。


**作业 1：补全最小循环的终止与回填**

在下面的 `TinyLoop.run` 里有两处填空：一处是动作执行后把观察结果回填进消息历史，一处是步数用尽时给出可读的失败信息。参考答案已经填在代码里，可以先在草稿上自己补全一遍，再运行对照。任务固定为"用工具算 7+8，再反转结果字符串"。

小提示：终止条件有两个——模型输出 `Final Answer`，或步数用尽；后者的提示要能看出任务没有完成。


In [ ]:
def reverse_brain(messages):
    """脚本化大脑：先调 calc 算 7+8，再调 reverse 反转结果字符串。"""
    issued = " ".join(m["content"] for m in messages)
    if "calc" not in issued:
        return 'Action: calc("7+8")'
    if "reverse" not in issued:
        return 'Action: reverse("15")'
    return "Final Answer: 51"


class TinyLoop:
    """极简循环：大脑、工具、消息历史三样。"""

    def __init__(self, brain, tools):
        self.brain = brain
        self.tools = tools
        self.messages = []

    def execute_tool(self, name, args):
        """在工具注册表里查找并执行工具，返回结果字符串。"""
        if name not in self.tools:
            return f"Error: 未知工具 {name}"
        return str(self.tools[name](args.strip().strip("\"'")))

    def run(self, task, max_steps=5):
        """运行循环，返回是否成功与完整的消息历史。"""
        self.messages = [{"role": "user", "content": task}]
        for _ in range(max_steps):
            reply = self.brain(self.messages)
            self.messages.append({"role": "assistant", "content": reply})
            kind, action, _ = parse_response(reply)
            if action is not None:
                name, args = action
                result = self.execute_tool(name, args)
                # 填空 1：把观察结果回填进消息历史
                self.messages.append({"role": "user",
                                      "content": f"Observation: {result}"})
            if kind == "final":
                return True, self.messages
        # 填空 2：步数用尽时给出可读的失败信息
        return False, self.messages + [{"role": "user",
                                        "content": "Error: 步数用尽，未得到最终答案"}]


tools = {"calc": calc, "reverse": lambda s: s[::-1]}
tiny = TinyLoop(brain=reverse_brain, tools=tools)
ok, trace = tiny.run("请用工具计算 7+8，再反转结果字符串", max_steps=5)

assert ok, "循环应在 5 步内给出 Final Answer"
assert trace[-1]["content"].startswith("Final Answer"), "最后一条消息应是最终答案"
assert sum(m["content"].startswith("Observation") for m in trace) == 2, "应有两条观察结果"
print("作业 1 通过：循环在 5 步内终止，中间结果被回填进消息历史")


**作业 2：动作解析器**

从零写一个更简的 `parse_action`，返回 `(kind, name, args)` 三元组：支持 `Action`、`Final Answer`、无动作三种情况，并正确处理多余空白与多行。参考答案已经填在代码里，可以先在草稿上自己补全一遍，再运行对照。

小提示：先找 Final Answer 再做 Action，因为 Final 所在行不该被当成 Action。


In [ ]:
import re


def parse_action(text):
    """把模型回复解析成 (kind, name, args)。

    kind 为 "final" 时 name 为 None、args 为答案；
    kind 为 "action" 时 name 为工具名、args 为参数；
    kind 为 "idle" 时 name 与 args 均为 None。
    """
    final = re.search(r"Final Answer:\s*(.+)", text, re.DOTALL)
    action = re.search(r"Action:\s*(\w+)\s*\((.*?)\)", text, re.DOTALL)
    if final:
        return ("final", None, final.group(1).strip())
    if action:
        return ("action", action.group(1), action.group(2).strip())
    return ("idle", None, None)


# 边界情况：多余空白、多行、Final 与 Action 同时出现
assert parse_action('Action: add(1, 2)') == ("action", "add", "1, 2")
assert parse_action('Final Answer: 42') == ("final", None, "42")
assert parse_action('  看看  这个  \n\n') == ("idle", None, None)
assert parse_action('Action:  add( 3 , 5 )') == ("action", "add", "3 , 5")
assert parse_action('Thought: 先想一下\nAction: search(x)')[0] == "action"
assert parse_action('Final Answer: 好\nAction: foo(1)')[0] == "final"
print("作业 2 通过：三种情况、多余空白、多行与优先级都能正确解析")


**作业 3：单次调用与循环的量化差异**

对同一任务分别执行单次调用与 Agent 循环。请先自己写 `count_tool_calls` 统计消息历史里的工具调用数，再运行对照。断言循环的工具调用次数大于 1 且上下文里包含工具返回值，而单次调用两者皆无。

小提示：工具返回值靠我们自己回填上下文，所以"循环拿到了工具结果"是可断言的，不依赖具体模型。


In [ ]:
def count_tool_calls(messages):
    """统计一段消息历史里执行过的工具动作数。"""
    # 填空：Observation 消息的条数就是工具调用次数
    return sum(1 for m in messages if m["content"].startswith("Observation"))


# 同一任务：单次调用 vs Agent 循环
single_reply = client.chat([{"role": "user", "content": "请分步计算 (3+5)×(7-2)"}])
single_trace = [{"role": "user", "content": single_reply}]

loop_tools = {"calc": calc}
loop = AgentLoop(brain=arithmetic_brain, tools=loop_tools)
loop_trace = loop.run("请分步计算 (3+5)×(7-2)", max_steps=6)

assert count_tool_calls(loop_trace) > 1, "循环应多次调用工具"
assert any(m["content"].startswith("Observation") for m in loop_trace), "循环应拿到工具返回值"
assert count_tool_calls(single_trace) == 0, "单次调用不应有工具调用"
print(f"单次调用返回：{single_reply}")
print("作业 3 通过：循环的工具调用次数大于 1，单次调用为 0")


## 参考资料

- Wooldridge & Jennings, [Intelligent Agents: Theory and Practice](https://www.csc.liv.ac.uk/~mjw/pubs/ker95.pdf), 1995 — Agent 经典定义的出处：situated、autonomous、感知—行动闭环
- Wooldridge, [Intelligent Agents（Multiagent Systems 第 1 章）](https://www.cs.ox.ac.uk/people/michael.wooldridge/pubs/maia-chapter.pdf), 1999 — 更完整的 agent 定义，含理性 agent 与 BDI 讨论
- Yao et al., [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629), 2022 — 推理与行动交替的范式，本课默认的 Agent 循环骨架
- Wang et al., [A Survey on LLM-based Autonomous Agents](https://arxiv.org/abs/2308.11432), 2023 — Agent 组成框架的共识总结：planning、memory、tool use
- Xi et al., [The Rise and Potential of Large Language Model Based Agents](https://arxiv.org/abs/2309.07864), 2023 — 更全面的 LLM-based Agent 综述与生态分类
- Weng, [LLM Powered Autonomous Agents](https://lilianweng.github.io/posts/2023-06-23-agent/), 2023 — 把 Agent 循环讲得最清晰的一篇博客，适合作为第一份扩展阅读
- Schick et al., [Toolformer: Language Models Can Teach Themselves to Use Tools](https://arxiv.org/abs/2302.04761), 2023 — LLM 学习调用工具的早期代表作
- Zhou et al., [Language Agent Tree Search Unifies Reasoning, Acting, and Planning](https://arxiv.org/abs/2310.04406), 2023 — 规划/搜索范式（LATS）
- Packer et al., [MemGPT: Towards LLMs as Operating Systems](https://arxiv.org/abs/2310.08560), 2023 — 记忆范式
- Wu et al., [AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation](https://arxiv.org/abs/2308.08155), 2023 — 多 Agent 协作框架
- Anthropic, [Model Context Protocol](https://modelcontextprotocol.io), 2024 — 工具与上下文的标准化接口，Agent 生态的互连协议
- Stanford, [CS329A Course Homepage](https://cs329a.stanford.edu/), 2025 — 本课课程大纲与作业来源
